# Weighted graphs для 368 Xela-сенсоров

Ноутбук строит три типа sensor-level графов по 3D координатам сенсоров: physical, distance threshold и kNN. Вес каждого ребра равен евклидову расстоянию между двумя сенсорами в метрах.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from omegaconf import OmegaConf

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **_: x

repo_root = Path.cwd()
if repo_root.name == "graph_visualizations":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from tactile_ssl.data.xela_tactile import XelaSSLDataset
from tactile_ssl.data.xela.utils import XELA_FLATTEN_ORDER
from tactile_ssl.graph.builders import (
    build_distance_threshold_graph,
    build_knn_graph,
    build_physical_graph,
    build_sensor_graph,
)
from tactile_ssl.graph.stats import sensor_graph_stats
from tactile_ssl.graph.utils import SENSOR_TO_HAND_PART, validate_against_xela_flatten_order
from tactile_ssl.graph.vis_utils import plot_sensor_graph_comparison

validate_against_xela_flatten_order(XELA_FLATTEN_ORDER)


## Параметры

In [ ]:
DATA_ROOT_OVERRIDE = None  # например: "/path/to/sparsh-skin-dataset"
LOCAL_DATASET_CANDIDATE = repo_root / "facebook--sparsh-skin-dataset"
SEQUENCE_NAME = "drill"
SEQUENCE_ID = 0

DISTANCE_THRESHOLD = 0.02  # meters
K_NEIGHBORS = 6
BRIDGE_K = 1
NUM_VIS_WINDOWS = 10
FRAME_IN_WINDOW = 0

print(f"Distance threshold: {DISTANCE_THRESHOLD:.3f} m = {DISTANCE_THRESHOLD * 100:.1f} cm")
print(f"k nearest neighbors: {K_NEIGHBORS}")
print(f"physical bridge_k: {BRIDGE_K}")


## Загрузка последовательности

In [ ]:
paths_cfg = OmegaConf.load(repo_root / "config" / "paths" / "default.yaml")
data_cfg = OmegaConf.load(repo_root / "config" / "data" / "xela.yaml")

if DATA_ROOT_OVERRIDE is not None:
    data_root = Path(DATA_ROOT_OVERRIDE).expanduser()
elif (LOCAL_DATASET_CANDIDATE / "xela" / "pretraining" / "extracted").exists():
    data_root = LOCAL_DATASET_CANDIDATE
else:
    data_root = Path(paths_cfg.data_root).expanduser()

extracted_root = data_root / "xela" / "pretraining" / "extracted"
sequence_path = extracted_root / SEQUENCE_NAME / str(SEQUENCE_ID)
baseline_signal_path = extracted_root / "baseline" / "xela" / "data.pkl"
urdf_path = extracted_root / "urdf" / "ahrcpcpn.urdf"

missing = [path for path in (sequence_path, baseline_signal_path, urdf_path) if not path.exists()]
if missing:
    missing_lines = "\n".join(f"- {path}" for path in missing)
    raise FileNotFoundError(
        "Не найдены файлы датасета для графовой визуализации. "
        "Переопределите DATA_ROOT_OVERRIDE на локальный корень sparsh-skin-dataset.\n"
        f"Проверенные пути:\n{missing_lines}"
    )

dataset_config = OmegaConf.create(
    {
        "window_time": data_cfg.window_time,
        "window_overlap": data_cfg.window_overlap,
        "interpolating_freq": data_cfg.interpolating_freq,
        "subtract_baseline": True,
        "smooth_data": False,
        "bias_noise_std": 0.0,
        "bias_range": 0.0,
        "cache": {"enabled": False, "root": str(repo_root / ".cache" / "xela_artifacts")},
        "preprocessing": data_cfg.preprocessing,
        "features": {"use_spatial_coords": False},
    }
)

dataset = XelaSSLDataset(
    config=dataset_config,
    data_path=str(sequence_path),
    xela_urdf_path=str(urdf_path),
    baseline_signal_path=str(baseline_signal_path),
    load_images=False,
)

print(f"data_root: {data_root}")
print(f"sequence_path: {sequence_path}")
print(f"windows: {len(dataset)}")
print(f"frames per window: {dataset.num_frames_per_window}")


## Helpers

In [ ]:
GRAPH_TYPES = ("physical", "distance_threshold", "knn")

def get_window_positions(window_id, frame_in_window=FRAME_IN_WINDOW):
    sample = dataset[window_id]
    frame_idx = min(frame_in_window, sample["sensor_poses"].shape[0] - 1)
    return sample["sensor_poses"][frame_idx, :, :3].detach().cpu().numpy()

def build_graph_by_type(graph_type, positions):
    if graph_type == "physical":
        return build_physical_graph(positions, bridge_k=BRIDGE_K)
    if graph_type == "distance_threshold":
        return build_distance_threshold_graph(positions, threshold=DISTANCE_THRESHOLD)
    if graph_type == "knn":
        return build_knn_graph(positions, k=K_NEIGHBORS, symmetrize=True)
    return build_sensor_graph(positions, graph_type)

def window_ids_for_visualization(num_windows=NUM_VIS_WINDOWS):
    if len(dataset) <= num_windows:
        return list(range(len(dataset)))
    return sorted(set(np.linspace(0, len(dataset) - 1, num_windows, dtype=int).tolist()))

VIS_WINDOW_IDS = window_ids_for_visualization()
VIS_POSITIONS = {window_id: get_window_positions(window_id) for window_id in VIS_WINDOW_IDS}
print(VIS_WINDOW_IDS)


## Метрики по всему датасету

In [ ]:
rows = []
for window_id in tqdm(range(len(dataset)), desc="Graph metrics"):
    positions = get_window_positions(window_id)
    for graph_type in GRAPH_TYPES:
        graph = build_graph_by_type(graph_type, positions)
        row = sensor_graph_stats(graph)
        row["window_id"] = window_id
        row["graph_type"] = graph_type
        rows.append(row)

metrics_df = pd.DataFrame(rows)
summary_df = metrics_df.groupby("graph_type").mean(numeric_only=True)
summary_df


In [ ]:
metrics_to_plot = ["num_edges", "avg_degree", "num_components", "mean_edge_weight"]
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(4.5 * len(metrics_to_plot), 4))
for ax, metric in zip(axes, metrics_to_plot):
    summary_df[metric].plot(kind="bar", ax=ax)
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


## Physical graph: 10 временных окон

In [ ]:
for window_id in VIS_WINDOW_IDS:
    positions = VIS_POSITIONS[window_id]
    graph = build_graph_by_type("physical", positions)
    print(f"window={window_id}", sensor_graph_stats(graph))
    fig = plot_sensor_graph_comparison(positions, graph)
    fig.suptitle(f"physical graph | window {window_id}", y=1.02)
    plt.show()


## Distance-threshold graph: 10 временных окон

In [ ]:
print(f"Distance threshold: {DISTANCE_THRESHOLD:.3f} m = {DISTANCE_THRESHOLD * 100:.1f} cm")
for window_id in VIS_WINDOW_IDS:
    positions = VIS_POSITIONS[window_id]
    graph = build_graph_by_type("distance_threshold", positions)
    print(f"window={window_id}", sensor_graph_stats(graph))
    fig = plot_sensor_graph_comparison(positions, graph)
    fig.suptitle(f"distance-threshold graph | window {window_id}", y=1.02)
    plt.show()


## kNN graph: 10 временных окон

In [ ]:
print(f"k nearest neighbors: {K_NEIGHBORS}")
for window_id in VIS_WINDOW_IDS:
    positions = VIS_POSITIONS[window_id]
    graph = build_graph_by_type("knn", positions)
    print(f"window={window_id}", sensor_graph_stats(graph))
    fig = plot_sensor_graph_comparison(positions, graph)
    fig.suptitle(f"kNN graph | window {window_id}", y=1.02)
    plt.show()
